# Character Consistency Test Template (FLUX.2 klein 4B, local)
Runs all 6 scenes of a story with a FIXED seed shared across scenes, then displays a comparison grid. This example uses the Chiku/Pinku story — swap the `scenes` list for Nischay or Ananya's story JSON to reuse for those. Set Runtime to T4 GPU first.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch
from diffusers import Flux2KleinPipeline
import time

MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"
SEED = 42  # SAME seed reused for every scene — the key consistency lever

pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
pipe.enable_model_cpu_offload()

In [ ]:
# Replace this list with scenes from Nischay.json or Ananya.json to test those stories
scenes = [
    {"scene_id": "s01", "prompt": "Chiku and Pinku starting their adventure in the garden. Style: Classic 1990s hand-drawn animated jungle film with clean ink outlines and cel shading. Chiku, the small sleek cat with soft gray fur. Pointed ears. Sharp green eyes. Long graceful tail. Four agile legs. Standing on grass. Pinku, the friendly dog with golden brown fur. Floppy ears. Bright loyal eyes. Wagging tail. Four legs. Running beside Chiku. Both characters visible. Jungle garden setting with lush green grass and bushes. Afternoon sunlight filtering through leaves. Light beige ground. The scene shows excitement and adventure."},
    {"scene_id": "s02", "prompt": "Bittu darting up a tree as Chiku and Pinku watch. Style: Classic 1990s hand-drawn animated jungle film with clean ink outlines and cel shading. Bittu, the squirrel with reddish-brown fur. Bushy tail. Small rounded ears. Bright eyes. Four legs. Sharp claws. Cheek pouches. Climbing tree. Chiku and Pinku watching from below. Both characters visible. Jungle garden with tall trees and green foliage. Afternoon sunlight. Light beige ground. The scene shows curiosity and observation."},
    {"scene_id": "s03", "prompt": "Chiku sitting on warm grass basking in sunshine. Style: Classic 1990s hand-drawn animated jungle film with clean ink outlines and cel shading. Chiku, the small sleek cat with soft gray fur. Pointed ears. Sharp green eyes. Long graceful tail. Four agile legs. Sitting on grass. Afternoon sunlight. Light beige ground. ONLY Chiku is visible in this scene. The scene shows relaxation and contentment."},
    {"scene_id": "s04", "prompt": "Pinku digging and running around in the garden. Style: Classic 1990s hand-drawn animated jungle film with clean ink outlines and cel shading. Pinku, the friendly dog with golden brown fur. Floppy ears. Bright loyal eyes. Wagging tail. Four legs. Digging in soil. Afternoon sunlight. Light beige ground. ONLY Pinku is visible in this scene. The scene shows energy and playfulness."},
    {"scene_id": "s05", "prompt": "Chiku, Pinku, and Bittu playing together in the garden. Style: Classic 1990s hand-drawn animated jungle film with clean ink outlines and cel shading. Chiku, the small sleek cat with soft gray fur. Pointed ears. Sharp green eyes. Long graceful tail. Four agile legs. Pinku, the friendly dog with golden brown fur. Floppy ears. Bright loyal eyes. Wagging tail. Four legs. Bittu, the squirrel with reddish-brown fur. Bushy tail. Small rounded ears. Bright eyes. Four legs. Sharp claws. Cheek pouches. All characters playing together. Jungle garden with green foliage. Afternoon sunlight. Light beige ground. The scene shows joy and friendship."},
    {"scene_id": "s06", "prompt": "Chiku and Pinku resting after a fun day in the garden. Style: Classic 1990s hand-drawn animated jungle film with clean ink outlines and cel shading. Chiku, the small sleek cat with soft gray fur. Pointed ears. Sharp green eyes. Long graceful tail. Four agile legs. Pinku, the friendly dog with golden brown fur. Floppy ears. Bright loyal eyes. Wagging tail. Four legs. Both characters resting on grass. Jungle garden setting with green foliage. Afternoon sunlight. Light beige ground. The scene shows tiredness and happiness."}
]

images = []
total_gen_time = 0
for scene in scenes:
    print(f"Generating {scene['scene_id']}...")
    start = time.time()
    img = pipe(prompt=scene["prompt"], height=1024, width=1024, num_inference_steps=4, guidance_scale=1.0, generator=torch.Generator(device="cuda").manual_seed(SEED)).images[0]
    elapsed = time.time() - start
    total_gen_time += elapsed
    fname = f"story_{scene['scene_id']}.png"
    img.save(fname)
    images.append((scene["scene_id"], img))
    print(f"Saved {fname} ({elapsed:.1f}s)")
print(f"\nTotal: {total_gen_time:.1f}s | Avg/scene: {total_gen_time/len(scenes):.1f}s")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, (scene_id, img) in zip(axes.flat, images):
    ax.imshow(img); ax.set_title(scene_id); ax.axis("off")
plt.tight_layout()
plt.savefig("story_comparison.png", dpi=150)
plt.show()